Calculation and Visualization of a Golden Day, which can be picked at the beginning and will then be exported with data that include all Proxies.

In [3]:
import os
import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
import warnings

warnings.filterwarnings('ignore')

# =========================================================
# 1. Konfiguration & Golden Day
# =========================================================
target_date = "2024-07-01"
base_dir = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab"
dom_dir = os.path.join(base_dir, "Daten (DOM)")

# Pfade
csv_file = os.path.join(base_dir, "new_Output", "Features_RH_PM25_NDVI_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv")
verkehr_file = os.path.join(base_dir, "Daten (Verkehrsmesswerte)", "Verkehrsdaten_mit_Koordinaten_merged_cars_n_trucks.csv")

# =========================================================
# 2. Modell trainieren (Complete Case)
# =========================================================
print(f"--- Starte Modellierung für den Golden Day: {target_date} ---")
df = pd.read_csv(csv_file, sep=";", decimal=",")
df['station_id'] = df['station_id'].astype(str).str.strip()

bad_sensors = ['5456', '15449', '46505', '9094']
df = df[~df['station_id'].isin(bad_sensors)].copy()

features = [
    'prox_kumulativ_car', 'prox_kumulativ_truck', 'IDW_car', 'IDW_truck',
    'icon_temp_2m', 'icon_wind_speed', 'icon_wind_direction',
    'icon_precip_12h', 'icon_relative_humidity',
    'MAIAC_AOD', 'S5P_NO2', 'S5P_UVAI', 'ERA5_BLH',
    'tpi_300m', 'tpi_1500m', 'ndvi'
]

df = df.dropna(subset=['PM25_Target'] + features)

# Baseline Calibration
mean_kopf = df[df['station_id'] == "1"]['PM25_Target'].mean()
mean_low = df[df['station_id'] != "1"]['PM25_Target'].mean()
df.loc[df['station_id'] != "1", 'PM25_Target'] += (mean_kopf - mean_low)

print(f"-> Trainiere RF-Modell mit {len(df)} lückenlosen Samples...")
rf_model = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=4, random_state=42, n_jobs=-1)
rf_model.fit(df[features], df['PM25_Target'])

# =========================================================
# 3. 100m Grid erstellen
# =========================================================
print("\nErstelle 100x100m Grid aus Referenz-TIF...")
tpi_300_path = os.path.join(dom_dir, "wuerzburg_tpi_300m.tif")

with rasterio.open(tpi_300_path) as ref_src:
    bounds = ref_src.bounds
    crs = ref_src.crs
    x_min, x_max = bounds.left, bounds.right
    y_min, y_max = bounds.bottom, bounds.top

resolution = 100
grid_x, grid_y = np.meshgrid(np.arange(x_min, x_max, resolution),
                             np.arange(y_max, y_min, -resolution))

flat_x, flat_y = grid_x.flatten(), grid_y.flatten()
coords_list = list(zip(flat_x, flat_y))
grid_df = pd.DataFrame({'x_utm': flat_x, 'y_utm': flat_y})

# =========================================================
# 4. Raster-Werte abfragen (inkl. Feuchtigkeit)
# =========================================================
print("\nLade dynamische und statische Raster-Daten...")

def extract_raster(tif_path):
    if not os.path.exists(tif_path):
        print(f"  [WARNUNG] Fehlt: {tif_path} -> Belege mit 0.0")
        return np.zeros(len(coords_list))
    with rasterio.open(tif_path) as src:
        vals = np.array([val[0] for val in src.sample(coords_list)], dtype=float)
        invalid = (vals == src.nodata) | (vals > 1e6) | (vals < -1e6) | np.isnan(vals)
        vals[invalid] = np.nanmedian(vals[~invalid]) if np.any(~invalid) else 0.0
        return vals

grid_df['tpi_300m'] = extract_raster(tpi_300_path)
grid_df['tpi_1500m'] = extract_raster(os.path.join(dom_dir, "wuerzburg_tpi_1500m.tif"))
grid_df['ndvi'] = extract_raster(os.path.join(base_dir, "Vorbereitete Daten für RF", "AOI_NDVI_S2_11082024.tif"))

tifs = {
    'ERA5_BLH': os.path.join(dom_dir, "ERA5_BLH", f"ERA5_BLH_{target_date}.tif"),
    'icon_temp_2m': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Temp_2m", f"ICON_D2_Temp_{target_date}.tif"),
    'icon_wind_speed': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Wind_Speed", f"ICON_D2_WindSpeed_{target_date}.tif"),
    'icon_wind_direction': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Wind_Direction", f"wind_dir_{target_date}.tif"),
    'icon_precip_12h': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Precipitation_12h", f"ICON_D2_Precip_12h_{target_date}.tif"),
    'icon_relative_humidity': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Relative_Humidity_2m", f"ICON_D2_RelHumid_{target_date}.tif"),
    'MAIAC_AOD': os.path.join(dom_dir, "MAIAC_AOD", f"MAIAC_AOD_{target_date}.tif"),
    'S5P_NO2': os.path.join(dom_dir, "S5P_NO2", f"S5P_NO2_{target_date}.tif"),
    'S5P_UVAI': os.path.join(dom_dir, "S5P_UVAI", f"S5P_UVAI_{target_date}.tif")
}

for feat, path in tifs.items():
    grid_df[feat] = extract_raster(path)

# =========================================================
# 5. Hyper-lokalen Verkehr für alle Pixel berechnen
# =========================================================
print("\nBerechne Verkehrsemissionen für jedes einzelne Pixel...")

df_traffic = pd.read_csv(verkehr_file, sep=";", decimal=",")
df_traffic["Time"] = pd.to_datetime(df_traffic["Time"], format="%d.%m.%Y %H:%M:%S", errors="coerce")
df_traffic_13 = df_traffic[(df_traffic["Time"].dt.hour == 13) &
                           (df_traffic["Time"].dt.strftime("%Y-%m-%d") == target_date)].copy()

for col in ["Lat", "Lon", "processed_car_vol", "processed_truck_vol"]:
    df_traffic_13[col] = df_traffic_13[col].astype(str).str.replace(",", ".", regex=False)
    df_traffic_13[col] = pd.to_numeric(df_traffic_13[col], errors="coerce")

df_traffic_13 = df_traffic_13.dropna(subset=["Lat", "Lon", "processed_car_vol"])

# Arrays für die Ergebnisse vorbereiten
cum_car = np.zeros(len(grid_df))
cum_truck = np.zeros(len(grid_df))
max_car = np.zeros(len(grid_df))
max_truck = np.zeros(len(grid_df))

# Koordinaten-Transformation (Lat/Lon -> UTM des Grids)
transformer = Transformer.from_crs("epsg:4326", crs, always_xy=True)
grid_x_arr, grid_y_arr = grid_df['x_utm'].values, grid_df['y_utm'].values

for _, t_stat in df_traffic_13.iterrows():
    # Verkehrsmessstelle ins Grid-System projizieren
    t_x, t_y = transformer.transform(t_stat['Lon'], t_stat['Lat'])

    # Euklidische Distanz in Metern (schneller als Haversine!)
    dist = np.sqrt((grid_x_arr - t_x)**2 + (grid_y_arr - t_y)**2)

    # IDW Formel anwenden
    idw_car = t_stat['processed_car_vol'] / ((dist + 1.0)**2)
    idw_truck = t_stat['processed_truck_vol'] / ((dist + 1.0)**2) if pd.notna(t_stat['processed_truck_vol']) else 0.0

    # Kumulativ aufaddieren
    cum_car += idw_car
    cum_truck += idw_truck

    # Maximalwert für IDW_max updaten
    max_car = np.maximum(max_car, idw_car)
    max_truck = np.maximum(max_truck, idw_truck)

grid_df['prox_kumulativ_car'] = cum_car
grid_df['prox_kumulativ_truck'] = cum_truck
grid_df['IDW_car'] = max_car
grid_df['IDW_truck'] = max_truck

# =========================================================
# 6. Prediction & Export
# =========================================================
print("\nBerechne finale PM2.5 Heatmap...")

grid_df.fillna(grid_df.median(), inplace=True)
grid_df['PM25_Pred'] = rf_model.predict(grid_df[features])

pm25_matrix = grid_df['PM25_Pred'].values.reshape(grid_y.shape)
output_tiff = os.path.join(base_dir, "new_Output", "Maps", f"PM25_Downscaling_Map_{target_date}.tif")
os.makedirs(os.path.dirname(output_tiff), exist_ok=True)

with rasterio.open(output_tiff, 'w', driver='GTiff', height=pm25_matrix.shape[0],
                   width=pm25_matrix.shape[1], count=1, dtype=pm25_matrix.dtype,
                   crs=crs, transform=rasterio.transform.from_origin(x_min, y_max, resolution, resolution)) as new_dataset:
    new_dataset.write(pm25_matrix, 1)

print(f"\nERFOLG! Hochauflösende Karte gespeichert unter:\n{output_tiff}")

--- Starte Modellierung für den Golden Day: 2024-07-01 ---
-> Trainiere RF-Modell mit 145 lückenlosen Samples...

Erstelle 100x100m Grid aus Referenz-TIF...

Lade dynamische und statische Raster-Daten...

Berechne Verkehrsemissionen für jedes einzelne Pixel...

Berechne finale PM2.5 Heatmap...

ERFOLG! Hochauflösende Karte gespeichert unter:
C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Maps\PM25_Downscaling_Map_2024-07-01.tif


In this part all cloudfree scenes the model gets trained and then a loop goes over all available scenes (44) to calculate a downscaled scene of the whole summer (01.05. - 31.08.2024)

Thereby a mean, min and max-scene is exported.

In [22]:
import os
import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
import warnings

warnings.filterwarnings('ignore')

# =========================================================
# 1. Konfiguration
# =========================================================
base_dir = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab"
dom_dir = os.path.join(base_dir, "Daten (DOM)")
csv_file = os.path.join(base_dir, "new_Output", "Features_RH_PM25_NDVI_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv")
verkehr_file = os.path.join(base_dir, "Daten (Verkehrsmesswerte)", "Verkehrsdaten_mit_Koordinaten_merged_cars_n_trucks.csv")

# =========================================================
# 2. Modell trainieren (wie gewohnt)
# =========================================================
print("--- Trainiere Modell für das Summer-Averaging ---")
df = pd.read_csv(csv_file, sep=";", decimal=",")
df['station_id'] = df['station_id'].astype(str).str.strip()
df = df[~df['station_id'].isin(['5456', '15449', '46505', '9094'])].copy()

features = [
    'IDW_car', 'IDW_truck',
    'icon_temp_2m', 'icon_wind_speed', 'icon_wind_direction',
    'icon_precip_12h', 'icon_relative_humidity',
    'MAIAC_AOD', 'S5P_NO2', 'S5P_UVAI', 'ERA5_BLH',
    'tpi_300m', 'tpi_1500m'
]

df = df.dropna(subset=['PM25_Target'] + features)

# Baseline Calibration
mean_kopf = df[df['station_id'] == "1"]['PM25_Target'].mean()
mean_low = df[df['station_id'] != "1"]['PM25_Target'].mean()
df.loc[df['station_id'] != "1", 'PM25_Target'] += (mean_kopf - mean_low)

rf_model = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=4, random_state=42, n_jobs=-1)
rf_model.fit(df[features], df['PM25_Target'])

# =========================================================
# 3. Das statische 100m Grid & Konstante Raster laden
# =========================================================
print("\nErstelle 100x100m Grid und lade statische Features...")
tpi_300_path = os.path.join(dom_dir, "wuerzburg_tpi_300m.tif")

with rasterio.open(tpi_300_path) as ref_src:
    bounds = ref_src.bounds
    crs = ref_src.crs
    x_min, x_max = bounds.left, bounds.right
    y_min, y_max = bounds.bottom, bounds.top

resolution = 100
grid_x, grid_y = np.meshgrid(np.arange(x_min, x_max, resolution), np.arange(y_max, y_min, -resolution))
flat_x, flat_y = grid_x.flatten(), grid_y.flatten()
coords_list = list(zip(flat_x, flat_y))

grid_df_static = pd.DataFrame({'x_utm': flat_x, 'y_utm': flat_y})

def extract_raster(tif_path):
    if not os.path.exists(tif_path):
        return np.full(len(coords_list), np.nan)
    with rasterio.open(tif_path) as src:
        vals = np.array([val[0] for val in src.sample(coords_list)], dtype=float)
        invalid = (vals == src.nodata) | (vals > 1e6) | (vals < -1e6) | np.isnan(vals)
        vals[invalid] = np.nanmedian(vals[~invalid]) if np.any(~invalid) else 0.0
        return vals

grid_df_static['tpi_300m'] = extract_raster(tpi_300_path)
grid_df_static['tpi_1500m'] = extract_raster(os.path.join(dom_dir, "wuerzburg_tpi_1500m.tif"))

# Verkehrsdaten vorab laden
df_traffic_all = pd.read_csv(verkehr_file, sep=";", decimal=",")
df_traffic_all["Time"] = pd.to_datetime(df_traffic_all["Time"], format="%d.%m.%Y %H:%M:%S", errors="coerce")
for col in ["Lat", "Lon", "processed_car_vol", "processed_truck_vol"]:
    df_traffic_all[col] = df_traffic_all[col].astype(str).str.replace(",", ".", regex=False)
    df_traffic_all[col] = pd.to_numeric(df_traffic_all[col], errors="coerce")
transformer = Transformer.from_crs("epsg:4326", crs, always_xy=True)

# =========================================================
# 4. SCHLEIFE ÜBER ALLE VERFÜGBAREN TAGE
# =========================================================
valid_dates = df['date'].unique() # Wir nehmen nur Tage, an denen das Modell trainiert wurde (wolkenfrei!)
all_daily_maps = []

print(f"\nStarte Massen-Inferenz für {len(valid_dates)} wolkenfreie Tage...")

for current_date in valid_dates:
    print(f" -> Berechne Karte für: {current_date}")
    grid_df = grid_df_static.copy()

    # 4.1 Dynamische Wetter- und Satellitendaten laden
    tifs = {
        'ERA5_BLH': os.path.join(dom_dir, "ERA5_BLH", f"ERA5_BLH_{current_date}.tif"),
        'icon_temp_2m': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Temp_2m", f"ICON_D2_Temp_{current_date}.tif"),
        'icon_wind_speed': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Wind_Speed", f"ICON_D2_WindSpeed_{current_date}.tif"),
        'icon_wind_direction': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Wind_Direction", f"wind_dir_{current_date}.tif"),
        'icon_precip_12h': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Precipitation_12h", f"ICON_D2_Precip_12h_{current_date}.tif"),
        'icon_relative_humidity': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Relative_Humidity_2m", f"ICON_D2_RelHumid_{current_date}.tif"),
        'MAIAC_AOD': os.path.join(dom_dir, "MAIAC_AOD", f"MAIAC_AOD_{current_date}.tif"),
        'S5P_NO2': os.path.join(dom_dir, "S5P_NO2", f"S5P_NO2_{current_date}.tif"),
        'S5P_UVAI': os.path.join(dom_dir, "S5P_UVAI", f"S5P_UVAI_{current_date}.tif")
    }

    for feat, path in tifs.items():
        grid_df[feat] = extract_raster(path)

    # 4.2 Verkehr für exakt diesen Tag (13:00 Uhr) berechnen
    df_traffic_13 = df_traffic_all[(df_traffic_all["Time"].dt.hour == 13) &
                                   (df_traffic_all["Time"].dt.strftime("%Y-%m-%d") == current_date)].copy()
    df_traffic_13 = df_traffic_13.dropna(subset=["Lat", "Lon", "processed_car_vol"])

    cum_car, cum_truck, max_car, max_truck = np.zeros(len(grid_df)), np.zeros(len(grid_df)), np.zeros(len(grid_df)), np.zeros(len(grid_df))
    grid_x_arr, grid_y_arr = grid_df['x_utm'].values, grid_df['y_utm'].values

    for _, t_stat in df_traffic_13.iterrows():
        t_x, t_y = transformer.transform(t_stat['Lon'], t_stat['Lat'])
        dist = np.sqrt((grid_x_arr - t_x)**2 + (grid_y_arr - t_y)**2)
        idw_car = t_stat['processed_car_vol'] / ((dist + 1.0)**2)
        idw_truck = t_stat['processed_truck_vol'] / ((dist + 1.0)**2) if pd.notna(t_stat['processed_truck_vol']) else 0.0

        cum_car += idw_car
        cum_truck += idw_truck
        max_car = np.maximum(max_car, idw_car)
        max_truck = np.maximum(max_truck, idw_truck)

    grid_df['IDW_car'] = max_car
    grid_df['IDW_truck'] = max_truck

# --- FIX FÜR DAS SPATIAL OVERFITTING (CLIPPING) ---
    # 1. Ermittle den maximalen IDW-Wert, den das Modell jemals im Training gesehen hat
    max_idw_car_train = df['IDW_car'].max()
    max_idw_truck_train = df['IDW_truck'].max()

    # 2. Schneide (Clip) die extremen Ausreißer im Grid auf dieses Maximum ab
    grid_df_daily['IDW_car'] = np.clip(grid_df_daily['IDW_car'], 0, max_idw_car_train)
    grid_df_daily['IDW_truck'] = np.clip(grid_df_daily['IDW_truck'], 0, max_idw_truck_train)
    # --------------------------------------------------

    # 3. Vorhersage und Rekord-Überprüfung (Dein bisheriger Code)
    grid_df_daily.fillna(grid_df_daily.median(), inplace=True)
    pm25_pred = rf_model.predict(grid_df_daily[features])

    # 4.3 Vorhersage für diesen Tag
    grid_df.fillna(grid_df.median(), inplace=True)
    pm25_pred = rf_model.predict(grid_df[features])
    all_daily_maps.append(pm25_pred.reshape(grid_y.shape))


# =========================================================
# 5. Averaging (Die Durchschnittskarte berechnen)
# =========================================================
print("\nBilde den Mittelwert über alle berechneten Tageskarten...")
summer_mean_matrix = np.mean(all_daily_maps, axis=0)

output_tiff = os.path.join(base_dir, "new_Output", "Maps", "PM25_Summer_Mean_Map_2024.tif")
os.makedirs(os.path.dirname(output_tiff), exist_ok=True)

with rasterio.open(output_tiff, 'w', driver='GTiff', height=summer_mean_matrix.shape[0],
                   width=summer_mean_matrix.shape[1], count=1, dtype=summer_mean_matrix.dtype,
                   crs=crs, transform=rasterio.transform.from_origin(x_min, y_max, resolution, resolution)) as new_dataset:
    new_dataset.write(summer_mean_matrix, 1)

print(f"\nERFOLG! Hochauflösende Sommer-Durchschnittskarte gespeichert unter:\n{output_tiff}")


# =========================================================
# 5. Maximum (Die Worst-Case Karte berechnen)
# =========================================================
print("\nBilde das Maximum über alle berechneten Tageskarten...")
summer_max_matrix = np.max(all_daily_maps, axis=0)

output_tiff = os.path.join(base_dir, "new_Output", "Maps", "PM25_Summer_Max_Map_2024.tif")
os.makedirs(os.path.dirname(output_tiff), exist_ok=True)

with rasterio.open(output_tiff, 'w', driver='GTiff', height=summer_max_matrix.shape[0],
                   width=summer_max_matrix.shape[1], count=1, dtype=summer_max_matrix.dtype,
                   crs=crs, transform=rasterio.transform.from_origin(x_min, y_max, resolution, resolution)) as new_dataset:
    new_dataset.write(summer_max_matrix, 1)

print(f"\nERFOLG! Hochauflösende Sommer-Maximum-Karte gespeichert unter:\n{output_tiff}")

# =========================================================
# 5. Minimum (Die Best-Case Karte berechnen)
# =========================================================
print("\nBilde das Minimum über alle berechneten Tageskarten...")
summer_min_matrix = np.min(all_daily_maps, axis=0)

output_tiff = os.path.join(base_dir, "new_Output", "Maps", "PM25_Summer_Min_Map_2024.tif")
os.makedirs(os.path.dirname(output_tiff), exist_ok=True)

with rasterio.open(output_tiff, 'w', driver='GTiff', height=summer_min_matrix.shape[0],
                   width=summer_min_matrix.shape[1], count=1, dtype=summer_min_matrix.dtype,
                   crs=crs, transform=rasterio.transform.from_origin(x_min, y_max, resolution, resolution)) as new_dataset:
    new_dataset.write(summer_min_matrix, 1)

'''
import numpy as np

# =========================================================
# Extremwert-Tracker initialisieren
# =========================================================
absolute_min = np.inf
absolute_max = -np.inf
date_of_min = ""
date_of_max = ""

print(f"\nScanne {len(valid_dates)} Tageskarten nach absoluten Extremen...")

for current_date in valid_dates:
    print(f" -> Prüfe Tag: {current_date}")
    grid_df_daily = grid_df_static.copy()

    # 1. Dynamische Wetter- und Satellitendaten laden
    tifs = {
        'ERA5_BLH': os.path.join(dom_dir, "ERA5_BLH", f"ERA5_BLH_{current_date}.tif"),
        'icon_temp_2m': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Temp_2m", f"ICON_D2_Temp_{current_date}.tif"),
        'icon_wind_speed': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Wind_Speed", f"ICON_D2_WindSpeed_{current_date}.tif"),
        'icon_wind_direction': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Wind_Direction", f"wind_dir_{current_date}.tif"),
        'icon_precip_12h': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Precipitation_12h", f"ICON_D2_Precip_12h_{current_date}.tif"),
        'icon_relative_humidity': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Relative_Humidity_2m", f"ICON_D2_RelHumid_{current_date}.tif"),
        'MAIAC_AOD': os.path.join(dom_dir, "MAIAC_AOD", f"MAIAC_AOD_{current_date}.tif"),
        'S5P_NO2': os.path.join(dom_dir, "S5P_NO2", f"S5P_NO2_{current_date}.tif"),
        'S5P_UVAI': os.path.join(dom_dir, "S5P_UVAI", f"S5P_UVAI_{current_date}.tif")
    }

    for feat, path in tifs.items():
        grid_df_daily[feat] = extract_raster(path)

    # 2. Verkehr für diesen spezifischen Tag berechnen
    df_traffic_13 = df_traffic_all[(df_traffic_all["Time"].dt.hour == 13) &
                                   (df_traffic_all["Time"].dt.strftime("%Y-%m-%d") == current_date)].copy()
    df_traffic_13 = df_traffic_13.dropna(subset=["Lat", "Lon", "processed_car_vol"])

    cum_car, cum_truck, max_car, max_truck = np.zeros(len(grid_df_daily)), np.zeros(len(grid_df_daily)), np.zeros(len(grid_df_daily)), np.zeros(len(grid_df_daily))
    grid_x_arr, grid_y_arr = grid_df_daily['x_utm'].values, grid_df_daily['y_utm'].values

    for _, t_stat in df_traffic_13.iterrows():
        t_x, t_y = transformer.transform(t_stat['Lon'], t_stat['Lat'])
        dist = np.sqrt((grid_x_arr - t_x)**2 + (grid_y_arr - t_y)**2)
        idw_car = t_stat['processed_car_vol'] / ((dist + 1.0)**2)
        idw_truck = t_stat['processed_truck_vol'] / ((dist + 1.0)**2) if pd.notna(t_stat['processed_truck_vol']) else 0.0

        cum_car += idw_car
        cum_truck += idw_truck
        max_car = np.maximum(max_car, idw_car)
        max_truck = np.maximum(max_truck, idw_truck)

    grid_df_daily['prox_kumulativ_car'] = cum_car
    grid_df_daily['prox_kumulativ_truck'] = cum_truck
    grid_df_daily['IDW_car'] = max_car
    grid_df_daily['IDW_truck'] = max_truck

    # 3. Vorhersage und Rekord-Überprüfung
    grid_df_daily.fillna(grid_df_daily.median(), inplace=True)
    pm25_pred = rf_model.predict(grid_df_daily[features])

    daily_min = np.min(pm25_pred)
    daily_max = np.max(pm25_pred)

    if daily_min < absolute_min:
        absolute_min = daily_min
        date_of_min = current_date

    if daily_max > absolute_max:
        absolute_max = daily_max
        date_of_max = current_date

    print(f"  -> Tages-Min: {daily_min:.2f} | Tages-Max: {daily_max:.2f}")

# =========================================================
# Finale Auswertung
# =========================================================
print("\n" + "="*50)
print("🏆 HISTORISCHE SOMMER-EXTREME 2024 🏆")
print("="*50)
print(f"Sauberster Pixel im gesamten Sommer:  {absolute_min:.2f} µg/m³ (Gemessen am {date_of_min})")
print(f"Schmutzigster Pixel im gesamten Sommer: {absolute_max:.2f} µg/m³ (Gemessen am {date_of_max})")
print("="*50)

'''

--- Trainiere Modell für das Summer-Averaging ---

Erstelle 100x100m Grid und lade statische Features...

Starte Massen-Inferenz für 44 wolkenfreie Tage...
 -> Berechne Karte für: 2024-05-02
 -> Berechne Karte für: 2024-05-09
 -> Berechne Karte für: 2024-05-10
 -> Berechne Karte für: 2024-05-12
 -> Berechne Karte für: 2024-05-13
 -> Berechne Karte für: 2024-05-14
 -> Berechne Karte für: 2024-05-20
 -> Berechne Karte für: 2024-05-26
 -> Berechne Karte für: 2024-06-07
 -> Berechne Karte für: 2024-06-08
 -> Berechne Karte für: 2024-06-12
 -> Berechne Karte für: 2024-06-18
 -> Berechne Karte für: 2024-06-24
 -> Berechne Karte für: 2024-06-25
 -> Berechne Karte für: 2024-06-26
 -> Berechne Karte für: 2024-06-27
 -> Berechne Karte für: 2024-06-29
 -> Berechne Karte für: 2024-07-08
 -> Berechne Karte für: 2024-07-09
 -> Berechne Karte für: 2024-07-14
 -> Berechne Karte für: 2024-07-15
 -> Berechne Karte für: 2024-07-16
 -> Berechne Karte für: 2024-07-17
 -> Berechne Karte für: 2024-07-18
 -> 

'\nimport numpy as np\n\n# =========================================================\n# Extremwert-Tracker initialisieren\n# =========================================================\nabsolute_min = np.inf\nabsolute_max = -np.inf\ndate_of_min = ""\ndate_of_max = ""\n\nprint(f"\nScanne {len(valid_dates)} Tageskarten nach absoluten Extremen...")\n\nfor current_date in valid_dates:\n    print(f" -> Prüfe Tag: {current_date}")\n    grid_df_daily = grid_df_static.copy()\n\n    # 1. Dynamische Wetter- und Satellitendaten laden\n    tifs = {\n        \'ERA5_BLH\': os.path.join(dom_dir, "ERA5_BLH", f"ERA5_BLH_{current_date}.tif"),\n        \'icon_temp_2m\': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Temp_2m", f"ICON_D2_Temp_{current_date}.tif"),\n        \'icon_wind_speed\': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Wind_Speed", f"ICON_D2_WindSpeed_{current_date}.tif"),\n        \'icon_wind_direction\': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Wind_Direction", f"wind_dir_{current_d

In [12]:
import pandas as pd

# =========================================================
# 1. Daten laden
# =========================================================
file_path = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Features_RH_PM25_NDVI_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv"

# Datensatz laden (wie gewohnt mit Semikolon und Komma)
df = pd.read_csv(file_path, sep=";", decimal=",")

## Multiplying "S5P" with 10 *6 to have more meaningful results
if 'S5P_NO2' in df.columns:
    df['S5P_NO2'] = df['S5P_NO2'] * 1e6
    # Spalte umbenennen, damit die neue Einheit in der Tabelle steht
    df.rename(columns={'S5P_NO2': 'S5P_NO2 (µmol/m²)'}, inplace=True)

# =========================================================
# 2. Features auswählen, die wir uns ansehen wollen
# =========================================================
features_to_check = [
    'icon_temp_2m',
    'S5P_NO2 (µmol/m²)',
    'icon_relative_humidity',
    'S5P_UVAI',
    'IDW_car',
    'ERA5_BLH',
    'icon_wind_speed',
    'icon_wind_direction',
    'MAIAC_AOD',
    'IDW_truck',
    'tpi_300m',
    'tpi_1500m',
    'icon_precip_12h'
]

# Zur Sicherheit: Nur Spalten nehmen, die auch im DataFrame existieren
available_features = [f for f in features_to_check if f in df.columns]

# =========================================================
# 3. Statistik berechnen (Min, Max, Mean)
# =========================================================
# .describe() berechnet alles, wir filtern uns nur Min, Max und Mean heraus
summary_df = df[available_features].describe().loc[['min', 'max', 'mean']]

# Transponieren (.T) dreht die Tabelle um, sodass sie im Terminal besser lesbar ist
summary_df = summary_df.T

# Ergebnisse runden für eine schönere Optik
summary_df = summary_df.round(4)

print("--- Werte-Bereiche (Range) im gesamten Sommer-Datensatz ---")
print(summary_df.to_string())

--- Werte-Bereiche (Range) im gesamten Sommer-Datensatz ---
                             min        max       mean
icon_temp_2m             12.2000    32.9000    22.7095
S5P_NO2 (µmol/m²)         6.4324    69.5561    32.7940
icon_relative_humidity   34.0000    92.0000    54.9282
S5P_UVAI                 -2.0629     0.9354    -0.6210
IDW_car                   0.4803     3.5576     1.6219
ERA5_BLH                325.3285  2218.5178  1301.9362
icon_wind_speed           0.0000     6.5833     2.7919
icon_wind_direction       6.0000   360.0000   195.5188
MAIAC_AOD                 0.0415     0.5490     0.2336
IDW_truck                 0.0112     0.3266     0.1159
tpi_300m                -31.0442    23.6525    -2.9250
tpi_1500m               -60.8930    24.3901   -24.0610
icon_precip_12h           0.0000    33.3000     1.0607


In [11]:
import os
import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer
from sklearn.ensemble import RandomForestRegressor
import warnings

warnings.filterwarnings('ignore')

# =========================================================
# 1. Pfade & Neuen Ordner definieren
# =========================================================
base_dir = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab"
dom_dir = os.path.join(base_dir, "Daten (DOM)")
csv_file = os.path.join(base_dir, "new_Output", "Features_RH_PM25_NDVI_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv")
verkehr_file = os.path.join(base_dir, "Daten (Verkehrsmesswerte)", "Verkehrsdaten_mit_Koordinaten_merged_cars_n_trucks.csv")

# NEU: Der dedizierte Export-Ordner für die Tageskarten
output_dir = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Maps"
os.makedirs(output_dir, exist_ok=True)
print(f"Export-Ordner ist bereit: {output_dir}")

# =========================================================
# 2. Modell trainieren (OHNE NDVI)
# =========================================================
print("\n--- 1. Lade Daten & Trainiere Modell ---")
df = pd.read_csv(csv_file, sep=";", decimal=",")
df['station_id'] = df['station_id'].astype(str).str.strip()

# Kaputte Sensoren filtern
bad_sensors = ['5456', '15449', '46505', '9094']
df = df[~df['station_id'].isin(bad_sensors)].copy()

# ACHTUNG: 'ndvi' wurde hier aus der Liste entfernt!
features = [
    'prox_kumulativ_car', 'prox_kumulativ_truck', 'IDW_car', 'IDW_truck',
    'icon_temp_2m', 'icon_wind_speed', 'icon_wind_direction',
    'icon_precip_12h', 'icon_relative_humidity',
    'MAIAC_AOD', 'S5P_NO2', 'S5P_UVAI', 'ERA5_BLH',
    'tpi_300m', 'tpi_1500m'
]

df = df.dropna(subset=['PM25_Target'] + features)

# Baseline Kalibrierung (Kopfklinik-Abgleich)
mean_kopf = df[df['station_id'] == "1"]['PM25_Target'].mean()
mean_low = df[df['station_id'] != "1"]['PM25_Target'].mean()
df.loc[df['station_id'] != "1", 'PM25_Target'] += (mean_kopf - mean_low)

rf_model = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=4, random_state=42, n_jobs=-1)
rf_model.fit(df[features], df['PM25_Target'])
print("Modell erfolgreich ohne NDVI trainiert.")

# =========================================================
# 3. Das 100m Grid aufspannen & statische Raster laden
# =========================================================
print("\n--- 2. Erstelle 100m Raster für Würzburg ---")
tpi_300_path = os.path.join(dom_dir, "wuerzburg_tpi_300m.tif")

with rasterio.open(tpi_300_path) as ref_src:
    bounds = ref_src.bounds
    crs = ref_src.crs
    x_min, x_max = bounds.left, bounds.right
    y_min, y_max = bounds.bottom, bounds.top

resolution = 100
grid_x, grid_y = np.meshgrid(np.arange(x_min, x_max, resolution), np.arange(y_max, y_min, -resolution))
flat_x, flat_y = grid_x.flatten(), grid_y.flatten()
coords_list = list(zip(flat_x, flat_y))

grid_df_static = pd.DataFrame({'x_utm': flat_x, 'y_utm': flat_y})

def extract_raster(tif_path):
    if not os.path.exists(tif_path):
        return np.zeros(len(coords_list))
    with rasterio.open(tif_path) as src:
        vals = np.array([val[0] for val in src.sample(coords_list)], dtype=float)
        invalid = (vals == src.nodata) | (vals > 1e6) | (vals < -1e6) | np.isnan(vals)
        vals[invalid] = np.nanmedian(vals[~invalid]) if np.any(~invalid) else 0.0
        return vals

# TPI Raster laden (NDVI wird hier nicht mehr extrahiert)
grid_df_static['tpi_300m'] = extract_raster(tpi_300_path)
grid_df_static['tpi_1500m'] = extract_raster(os.path.join(dom_dir, "wuerzburg_tpi_1500m.tif"))

# Verkehrsdaten vorab in den Speicher laden
df_traffic_all = pd.read_csv(verkehr_file, sep=";", decimal=",")
df_traffic_all["Time"] = pd.to_datetime(df_traffic_all["Time"], format="%d.%m.%Y %H:%M:%S", errors="coerce")
for col in ["Lat", "Lon", "processed_car_vol", "processed_truck_vol"]:
    df_traffic_all[col] = df_traffic_all[col].astype(str).str.replace(",", ".", regex=False)
    df_traffic_all[col] = pd.to_numeric(df_traffic_all[col], errors="coerce")
transformer = Transformer.from_crs("epsg:4326", crs, always_xy=True)

# =========================================================
# 4. SCHLEIFE: Export jedes einzelnen Tages
# =========================================================
valid_dates = df['date'].unique()
print(f"\n--- 3. Starte Karten-Export für {len(valid_dates)} valide Tage ---")

for current_date in valid_dates:
    print(f" -> Berechne und exportiere: {current_date}")
    grid_df = grid_df_static.copy()

    # 4.1 Dynamische Wetter- und Satellitendaten laden
    tifs = {
        'ERA5_BLH': os.path.join(dom_dir, "ERA5_BLH", f"ERA5_BLH_{current_date}.tif"),
        'icon_temp_2m': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Temp_2m", f"ICON_D2_Temp_{current_date}.tif"),
        'icon_wind_speed': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Wind_Speed", f"ICON_D2_WindSpeed_{current_date}.tif"),
        'icon_wind_direction': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Wind_Direction", f"wind_dir_{current_date}.tif"),
        'icon_precip_12h': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Precipitation_12h", f"ICON_D2_Precip_12h_{current_date}.tif"),
        'icon_relative_humidity': os.path.join(dom_dir, "New_ICON_D2_Rasters", "Relative_Humidity_2m", f"ICON_D2_RelHumid_{current_date}.tif"),
        'MAIAC_AOD': os.path.join(dom_dir, "MAIAC_AOD", f"MAIAC_AOD_{current_date}.tif"),
        'S5P_NO2': os.path.join(dom_dir, "S5P_NO2", f"S5P_NO2_{current_date}.tif"),
        'S5P_UVAI': os.path.join(dom_dir, "S5P_UVAI", f"S5P_UVAI_{current_date}.tif")
    }

    for feat, path in tifs.items():
        grid_df[feat] = extract_raster(path)

    # 4.2 Verkehr für exakt diesen Tag (13:00 Uhr)
    df_traffic_13 = df_traffic_all[(df_traffic_all["Time"].dt.hour == 13) &
                                   (df_traffic_all["Time"].dt.strftime("%Y-%m-%d") == current_date)].copy()
    df_traffic_13 = df_traffic_13.dropna(subset=["Lat", "Lon", "processed_car_vol"])

    cum_car, cum_truck, max_car, max_truck = np.zeros(len(grid_df)), np.zeros(len(grid_df)), np.zeros(len(grid_df)), np.zeros(len(grid_df))
    grid_x_arr, grid_y_arr = grid_df['x_utm'].values, grid_df['y_utm'].values

    for _, t_stat in df_traffic_13.iterrows():
        t_x, t_y = transformer.transform(t_stat['Lon'], t_stat['Lat'])
        dist = np.sqrt((grid_x_arr - t_x)**2 + (grid_y_arr - t_y)**2)
        idw_car = t_stat['processed_car_vol'] / ((dist + 1.0)**2)
        idw_truck = t_stat['processed_truck_vol'] / ((dist + 1.0)**2) if pd.notna(t_stat['processed_truck_vol']) else 0.0

        cum_car += idw_car
        cum_truck += idw_truck
        max_car = np.maximum(max_car, idw_car)
        max_truck = np.maximum(max_truck, idw_truck)

    grid_df['prox_kumulativ_car'] = cum_car
    grid_df['prox_kumulativ_truck'] = cum_truck
    grid_df['IDW_car'] = max_car
    grid_df['IDW_truck'] = max_truck

    # 4.3 Vorhersage für diesen Tag
    grid_df.fillna(grid_df.median(), inplace=True)
    pm25_pred = rf_model.predict(grid_df[features])
    pm25_matrix = pm25_pred.reshape(grid_y.shape)

    # 4.4 Direkter Export in den neuen Ordner
    day_tiff = os.path.join(output_dir, f"PM25_Wuerzburg_{current_date}.tif")
    with rasterio.open(day_tiff, 'w', driver='GTiff', height=pm25_matrix.shape[0],
                       width=pm25_matrix.shape[1], count=1, dtype=pm25_matrix.dtype,
                       crs=crs, transform=rasterio.transform.from_origin(x_min, y_max, resolution, resolution)) as new_dataset:
        new_dataset.write(pm25_matrix, 1)

print("\n" + "="*50)
print(f"ALLE KARTEN ERFOLGREICH EXPORTIERT!")
print(f"Du findest sie hier: {output_dir}")
print("="*50)

Export-Ordner ist bereit: C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Maps

--- 1. Lade Daten & Trainiere Modell ---
Modell erfolgreich ohne NDVI trainiert.

--- 2. Erstelle 100m Raster für Würzburg ---

--- 3. Starte Karten-Export für 44 valide Tage ---
 -> Berechne und exportiere: 2024-05-02
 -> Berechne und exportiere: 2024-05-09
 -> Berechne und exportiere: 2024-05-10
 -> Berechne und exportiere: 2024-05-12
 -> Berechne und exportiere: 2024-05-13
 -> Berechne und exportiere: 2024-05-14
 -> Berechne und exportiere: 2024-05-20
 -> Berechne und exportiere: 2024-05-26
 -> Berechne und exportiere: 2024-06-07
 -> Berechne und exportiere: 2024-06-08
 -> Berechne und exportiere: 2024-06-12
 -> Berechne und exportiere: 2024-06-18
 -> Berechne und exportiere: 2024-06-24
 -> Berechne und exportiere: 2024-06-25
 -> Berechne und exportiere: 2024-06-26
 -> Berechne und exportiere: 2024-06-27
 -> Berechne und exportiere: 2024-06-29
 -> Berechne und exportiere: 2024-07-08
 -> Berechn

In [12]:
import os
import glob
import numpy as np
import pandas as pd
import rasterio

# =========================================================
# 1. Pfade definieren
# =========================================================
input_dir = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Maps\Einzeltage"
output_dir = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Maps"

os.makedirs(output_dir, exist_ok=True)

# =========================================================
# 2. Dateien scannen und sortieren
# =========================================================
tif_files = glob.glob(os.path.join(input_dir, "PM25_Wuerzburg_*.tif"))
print(f"Starte Analyse: {len(tif_files)} Tageskarten gefunden.")

weekday_maps = []
weekend_maps = []
raster_meta = None

for tif in tif_files:
    # Datum aus dem Dateinamen isolieren (z.B. "2024-05-02")
    filename = os.path.basename(tif)
    date_str = filename.replace("PM25_Wuerzburg_", "").replace(".tif", "")

    # Wochentag bestimmen (0=Montag, 4=Freitag, 5=Samstag, 6=Sonntag)
    day_idx = pd.to_datetime(date_str).weekday()

    with rasterio.open(tif) as src:
        # Metadaten (CRS, Ausdehnung) beim ersten Bild merken
        if raster_meta is None:
            raster_meta = src.profile

        data = src.read(1)

        # Sortieren: Wochentag oder Wochenende?
        if day_idx < 5:
            weekday_maps.append(data)
        else:
            weekend_maps.append(data)

# =========================================================
# 3. Durchschnitt berechnen (Averaging)
# =========================================================
print(f"-> Berechne Durchschnitt für {len(weekday_maps)} Wochentage (Mo-Fr)...")
weekday_mean_matrix = np.mean(weekday_maps, axis=0)

print(f"-> Berechne Durchschnitt für {len(weekend_maps)} Wochenend-Tage (Sa-So)...")
weekend_mean_matrix = np.mean(weekend_maps, axis=0)

# =========================================================
# 4. Export der finalen Karten
# =========================================================
# Wochentags-Karte speichern
weekday_out = os.path.join(output_dir, "PM25_Wuerzburg_Mean_Weekdays.tif")
with rasterio.open(weekday_out, 'w', **raster_meta) as dst:
    dst.write(weekday_mean_matrix, 1)

# Wochenend-Karte speichern
weekend_out = os.path.join(output_dir, "PM25_Wuerzburg_Mean_Weekends.tif")
with rasterio.open(weekend_out, 'w', **raster_meta) as dst:
    dst.write(weekend_mean_matrix, 1)

print("\n" + "="*50)
print("ERFOLG! Die aggregierten Karten liegen bereit:")
print(f"1. {weekday_out}")
print(f"2. {weekend_out}")
print("="*50)

Starte Analyse: 44 Tageskarten gefunden.
-> Berechne Durchschnitt für 32 Wochentage (Mo-Fr)...
-> Berechne Durchschnitt für 12 Wochenend-Tage (Sa-So)...

ERFOLG! Die aggregierten Karten liegen bereit:
1. C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Maps\PM25_Wuerzburg_Mean_Weekdays.tif
2. C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Maps\PM25_Wuerzburg_Mean_Weekends.tif


In [21]:
import os
import numpy as np
import geopandas as gpd
import rasterio
from shapely.geometry import Point

# =========================================================
# 1. Pfade definieren
# =========================================================
base_dir = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab"
tpi_300_path = os.path.join(base_dir, "Daten (DOM)", "wuerzburg_tpi_300m.tif")

# Der Output-Pfad für QGIS
output_gpkg = os.path.join(base_dir, "new_Output", "Maps", "Wuerzburg_100m_Grid.gpkg")

# =========================================================
# 2. Ausdehnung und Raster aufbauen
# =========================================================
print("Lese räumliche Ausdehnung aus dem TPI-Raster...")
with rasterio.open(tpi_300_path) as ref_src:
    bounds = ref_src.bounds
    crs = ref_src.crs
    x_min, x_max = bounds.left, bounds.right
    y_min, y_max = bounds.bottom, bounds.top

print("Generiere 100m x 100m Netz...")
resolution = 100
grid_x, grid_y = np.meshgrid(
    np.arange(x_min, x_max, resolution),
    np.arange(y_max, y_min, -resolution)
)

# =========================================================
# 3. Punkte erstellen und als GeoPackage speichern
# =========================================================
# Verwandle die X/Y Koordinaten in echte GIS-Punkte
geometry = [Point(x, y) for x, y in zip(grid_x.flatten(), grid_y.flatten())]

# GeoDataFrame erstellen
gdf = gpd.GeoDataFrame({'Pixel_ID': range(len(geometry))}, geometry=geometry, crs=crs)

# Speichern
gdf.to_file(output_gpkg, driver="GPKG")

print("\n" + "="*50)
print(f"ERFOLG! Das Grid mit {len(gdf)} Punkten wurde exportiert.")
print(f"Du kannst diese Datei jetzt einfach in QGIS ziehen:")
print(output_gpkg)
print("="*50)

Lese räumliche Ausdehnung aus dem TPI-Raster...
Generiere 100m x 100m Netz...

ERFOLG! Das Grid mit 6000 Punkten wurde exportiert.
Du kannst diese Datei jetzt einfach in QGIS ziehen:
C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Maps\Wuerzburg_100m_Grid.gpkg
